In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\kaanb\Desktop\FLY\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documents = [
    {
        "id": "doc1",
        "text": "Hacettepe Üniversitesi, Türkiye'nin başkenti Ankara'da bulunan bir devlet üniversitesidir."
    },
    {
        "id": "doc2",
        "text": "Ankara'nın ana havalimanı Esenboğa Havalimanı'dır."
    },
    {
        "id": "doc3",
        "text": "Esenboğa Havalimanı 1955 yılında hizmete açılmıştır."
    },
    {
        "id": "doc4",
        "text": "İstanbul Havalimanı 2018 yılında açılmıştır."
    },
    {
        "id": "doc5",
        "text": "İzmir Adnan Menderes Havalimanı, İzmir'de bulunan önemli bir havalimanıdır."
    },
    {
        "id": "doc6",
        "text": "ODTÜ de Ankara'da bulunan bir üniversitedir."
    }
]

In [3]:
class EmbeddingRetriever:
    def __init__(self, documents, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.documents = documents
        self.model = SentenceTransformer(model_name)

        self.doc_texts = [doc["text"] for doc in documents]
        self.doc_embeddings = self.model.encode(self.doc_texts)

    def search(self, query, top_k=2):
        query_embedding = self.model.encode([query])
        scores = cosine_similarity(query_embedding, self.doc_embeddings)[0]

        ranked_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in ranked_indices:
            results.append({
                "id": self.documents[idx]["id"],
                "text": self.documents[idx]["text"],
                "score": float(scores[idx])
            })

        return results

In [4]:
def llm_rewrite_to_subqueries(original_query):
    """
    Bu fonksiyon LLM'in karmaşık sorguyu alt sorgulara böldüğünü temsil eder.
    """
    sub_queries = [
        "Hacettepe Üniversitesi hangi şehirde bulunmaktadır?",
        "Ankara'nın ana havalimanı hangisidir ve hangi yılda açılmıştır?"
    ]

    return sub_queries
def extract_intermediate_answer(evidence):
    """
    Demo amaçlı basit ara cevap çıkarımı.
    Gerçek sistemde bu adım LLM ile yapılabilir.
    """
    combined_text = " ".join([item["text"] for item in evidence])

    if "Ankara" in combined_text:
        return "Ankara"

    return "Bilinmiyor"
def generate_final_answer(original_query, all_evidence):
    """
    Demo amaçlı final cevap üretimi.
    Gerçek sistemde burası LLM cevap üretimi olabilir.
    """
    evidence_text = " ".join([item["text"] for item in all_evidence])

    city = "Ankara" if "Ankara" in evidence_text else "Bilinmiyor"
    airport = "Esenboğa Havalimanı" if "Esenboğa Havalimanı" in evidence_text else "Bilinmiyor"
    year = "1955" if "1955" in evidence_text else "Bilinmiyor"

    return (
        f"Hacettepe Üniversitesi'nin bulunduğu şehir {city}'dır. "
        f"{city}'nın ana havalimanı {airport}'dır ve {year} yılında hizmete açılmıştır."
    )

In [7]:
def run_demo():
    original_query = "Hacettepe Üniversitesi'nin bulunduğu şehirdeki ana havalimanı hangi yılda açılmıştır?"

    print("\n==============================")
    print("ORIGINAL QUERY")
    print("==============================")
    print(original_query)

    retriever = EmbeddingRetriever(documents)

    # -------------------------
    # Single-step retrieval
    # -------------------------

    print("\n==============================")
    print("SINGLE-STEP RETRIEVAL")
    print("==============================")
    print("Tek adımda orijinal soru embedding retriever'a veriliyor.\n")

    single_step_results = retriever.search(original_query, top_k=3)

    for i, result in enumerate(single_step_results, start=1):
        print(f"{i}. [{result['id']}] Score: {result['score']:.4f}")
        print(f"   {result['text']}")

    # -------------------------
    # LLM rewrite
    # -------------------------

    print("\n==============================")
    print("LLM REWRITE INTO SUB-QUERIES")
    print("==============================")

    sub_queries = llm_rewrite_to_subqueries(original_query)

    for i, sq in enumerate(sub_queries, start=1):
        print(f"Sub-query {i}: {sq}")

    # -------------------------
    # Multi-hop retrieval step 1
    # -------------------------

    print("\n==============================")
    print("TWO-STEP MULTI-HOP RETRIEVAL")
    print("==============================")
    print("Her iki adımda da aynı embedding retriever kullanılıyor.\n")

    print("STEP 1 QUERY:")
    print(sub_queries[0])

    evidence_1 = retriever.search(sub_queries[0], top_k=2)

    print("\nSTEP 1 EVIDENCE:")
    for i, result in enumerate(evidence_1, start=1):
        print(f"{i}. [{result['id']}] Score: {result['score']:.4f}")
        print(f"   {result['text']}")

    intermediate_answer = extract_intermediate_answer(evidence_1)

    print("\nINTERMEDIATE ANSWER:")
    print(intermediate_answer)

    # -------------------------
    # Multi-hop retrieval step 2
    # -------------------------

    step_2_query = f"{intermediate_answer}'nın ana havalimanı hangisidir ve hangi yılda açılmıştır?"

    print("\nSTEP 2 QUERY:")
    print(step_2_query)

    evidence_2 = retriever.search(step_2_query, top_k=4)

    print("\nSTEP 2 EVIDENCE:")
    for i, result in enumerate(evidence_2, start=1):
        print(f"{i}. [{result['id']}] Score: {result['score']:.4f}")
        print(f"   {result['text']}")

    # -------------------------
    # Evidence aggregation
    # -------------------------

    all_evidence = evidence_1 + evidence_2

    final_answer = generate_final_answer(original_query, all_evidence)

    print("\n==============================")
    print("FINAL ANSWER")
    print("==============================")
    print(final_answer)

    print("\n==============================")
    print("SUMMARY")
    print("==============================")
    print("Single-step retrieval: Orijinal soru tek seferde arandı.")
    print("Two-step retrieval: Soru LLM ile alt sorgulara bölündü.")
    print("Retriever: Her iki adımda da aynı embedding retriever kullanıldı.")
    print("Evidence aggregation: İki adımdan gelen kanıtlar birleştirildi.")


if __name__ == "__main__":
    run_demo()


ORIGINAL QUERY
Hacettepe Üniversitesi'nin bulunduğu şehirdeki ana havalimanı hangi yılda açılmıştır?


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12086.46it/s]



SINGLE-STEP RETRIEVAL
Tek adımda orijinal soru embedding retriever'a veriliyor.

1. [doc3] Score: 0.6046
   Esenboğa Havalimanı 1955 yılında hizmete açılmıştır.
2. [doc2] Score: 0.5916
   Ankara'nın ana havalimanı Esenboğa Havalimanı'dır.
3. [doc4] Score: 0.5133
   İstanbul Havalimanı 2018 yılında açılmıştır.

LLM REWRITE INTO SUB-QUERIES
Sub-query 1: Hacettepe Üniversitesi hangi şehirde bulunmaktadır?
Sub-query 2: Ankara'nın ana havalimanı hangisidir ve hangi yılda açılmıştır?

TWO-STEP MULTI-HOP RETRIEVAL
Her iki adımda da aynı embedding retriever kullanılıyor.

STEP 1 QUERY:
Hacettepe Üniversitesi hangi şehirde bulunmaktadır?

STEP 1 EVIDENCE:
1. [doc1] Score: 0.6440
   Hacettepe Üniversitesi, Türkiye'nin başkenti Ankara'da bulunan bir devlet üniversitesidir.
2. [doc6] Score: 0.4467
   ODTÜ de Ankara'da bulunan bir üniversitedir.

INTERMEDIATE ANSWER:
Ankara

STEP 2 QUERY:
Ankara'nın ana havalimanı hangisidir ve hangi yılda açılmıştır?

STEP 2 EVIDENCE:
1. [doc2] Score: 0.8070
   A